In [ ]:
import logging
import os
import subprocess
import argparse
from datetime import datetime
from enum import Enum

import requests
from astropy.time import Time
from rich.console import Console
from rich.logging import RichHandler
from rich.table import Table
from rich import box
from rich import print as rprint
import pandas as pd

from soarpy import lco_client

In [ ]:
class Constants:
    LCO_FRAME_URL = 'https://archive-api.lco.global/frames/'
    SOAR_TELESCOPE_ID = '4m0a'
    QUERY_LIMIT = 200
    OBSMODE_INSTRUMENTS = {
    'photometry': {'ghts_blue_imager', 'ghts_red_imager'},
    'spectroscopy': {'ghts_blue', 'ghts_red'}
    }
    REQUIRED_CCD_BINNING = "2x2"

In [ ]:
client = lco_client.LCOClient()

In [ ]:
frames = client.get_frames("2026-06-21T00:00:00Z", "2026-06-23T00:00:00Z")

In [ ]:
#phot bias

phot_bias_frames = client._get_photometric_bias_frames(frames)
print(len(phot_bias_frames))

for frame in phot_bias_frames:
    print(frame["filename"])

In [ ]:
#phot flat
phot_flat_frames = client._get_photometric_flat_frames(frames)
print(len(phot_flat_frames))

for frame in phot_flat_frames:
    print(frame["filename"])

In [ ]:
#phot science
phot_science_frames = client._get_photometric_science_frames(frames)
print(len(phot_science_frames))

for frame in phot_science_frames:
    print(frame["filename"])

In [ ]:
#spec bias

spec_bias_frames = client._get_spectroscopic_bias_frames(frames)
print(len(spec_bias_frames))

for frame in spec_bias_frames:
    print(frame["filename"], frame["target_name"])

In [ ]:
#spac flat

spec_flats = client._get_spectroscopic_flat_frames(frames)
print(len(spec_flats))

for frame in spec_flats:
    print(frame["filename"], frame["target_name"])

In [ ]:
#spec arc

spec_arcs = client._get_spectroscopic_arc_frames(frames)
print(len(spec_arcs))

for frame in spec_arcs:
    print(frame["filename"], frame["target_name"])

In [ ]:
#spec standard

spec_standards = client._get_standard_star_spectroscopic_frames(frames)
print(len(spec_standards))

for frame in spec_standards:
    print(frame["filename"], frame["target_name"])

In [ ]:
#spec sciece

spec_science_frames = client._get_spectroscopic_science_frames(frames)
print(len(spec_science_frames))

for frame in spec_science_frames:
    print(frame["filename"], frame["target_name"])

In [ ]:
spec_science_frames[0]

In [ ]:
test_photometric_science_frames = client._get_spectroscopic_science_frames(frames)
len(test_photometric_science_frames)

In [ ]:
test_phtometric_flat_frames = client._get_photometric_flat_frames(frames)
len(test_phtometric_flat_frames)

In [ ]:
test_standard_frames = client._get_standard_star_spectroscopic_frames(frames)
len(test_standard_frames)

for t in test_standard_frames:
    print(t['filename'])

In [ ]:
test_arc_frames = client._get_spectroscopic_arc_frames(frames)
len(test_arc_frames)

for t in test_arc_frames:
    print(t['filename'])

In [ ]:
for t in test_phtometric_flat_frames:
    print(t['filename'])

In [ ]:
for t in test_photometric_science_frames:
    print(t["filename"])

In [ ]:
phot_flat_frames[0]

In [ ]:
unique_targets = set([frame["target_name"] for frame in frames])
unique_proposals = set([frame["proposal_id"] for frame in frames])
unique_configurations = set([frame["configuration_type"] for frame in frames])
unique_obstypes = set([frame["OBSTYPE"] for frame in frames])

rprint(f"Unique targets: {unique_targets}")
rprint(f"Unique proposals: {unique_proposals}")
rprint(f"Unique configurations: {unique_configurations}")
rprint(f"Unique OBSTYPES: {unique_obstypes}")

In [ ]:
#lets check what is there is calibrate proposal id:

calibration_frames = [frame for frame in frames if frame["proposal_id"] == "calibrate"]
rprint(f"total frames: {len(frames)}")
rprint(f"Calibration frames: {len(calibration_frames)}")

In [ ]:
unique_configurations = set([frame["configuration_type"] for frame in calibration_frames])
rprint(f"Unique configurations in calibration frames: {unique_configurations}")

In [ ]:
targets = set([frame['target_name'] for frame in frames])
insts = set([frame['instrument_id'] for frame in frames])
rprint(f"Unique targets: {targets}")

In [ ]:
insts = set([frame['configuration_type'] for frame in calibration_frames])
rprint(f"Unique instruments in calibration frames: {insts}")

In [ ]:
phot_frames = []
for frame in frames:
    if frame["instrument_id"].lower() in Constants.OBSMODE_INSTRUMENTS['photometry']:
        phot_frames.append(frame)

In [ ]:
unique_targets = set([frame["target_name"] for frame in phot_frames])
unique_proposals = set([frame["proposal_id"] for frame in phot_frames])
unique_configurations = set([frame["configuration_type"] for frame in phot_frames])
unique_obstypes = set([frame["OBSTYPE"] for frame in phot_frames])

rprint(f"Unique targets: {unique_targets}")
rprint(f"Unique proposals: {unique_proposals}")
rprint(f"Unique configurations: {unique_configurations}")
rprint(f"Unique OBSTYPES: {unique_obstypes}")

In [ ]:
#instrument in photometry, proposal in calibrate, 
# Photometric bias = instrument in photometry, proposal in calibrate, configuration_type is bias, rlevel = 0
# photometric flat = instrument in photometry, proposal in calibrate, configuration_type is lampflat, rlevel = 0
# photometric science = instrument in photometry, proposal not in calibrate, configuration_type is science. rlevel = 0

In [ ]:
def _get_photometric_bias_frames(frames):
    return [frame for frame in frames if frame["instrument_id"].lower() in Constants.OBSMODE_INSTRUMENTS['photometry'] and frame["proposal_id"] == "calibrate" and frame["configuration_type"].lower() == "bias"]

def _get_photometric_flat_frames(frames):
    return [frame for frame in frames if frame["instrument_id"].lower() in Constants.OBSMODE_INSTRUMENTS['photometry'] and frame["proposal_id"] == "calibrate" and frame["configuration_type"].lower() == "lampflat"]

def _get_photometric_science_frames(frames):
    return [frame for frame in frames if frame["instrument_id"].lower() in Constants.OBSMODE_INSTRUMENTS['photometry'] and frame["proposal_id"] != "calibrate" and frame["configuration_type"].lower() == "expose"]

In [ ]:
bias, flat, science = _get_photometric_bias_frames(frames), _get_photometric_flat_frames(frames), _get_photometric_science_frames(frames)

print(f"Photometric bias frames: {len(bias)}")
print(f"Photometric flat frames: {len(flat)}")
print(f"Photometric science frames: {len(science)}")

In [ ]:
# lets do same thing for spectroscopy frames:
#spectroscopic bias = instrument in spectroscopy, proposal in calibrate, configuration_type is bias, rlevel = 0
# spectroscopic flat = instrument in spectroscopy, proposal in calibrate, configuration_type is lampflat, rlevel = 0
# spectroscopic science = instrument in spectroscopy, proposal not in calibrate, configuration_type is science. rlevel = 0
#spectroscopic_arc = instrument in spectroscopy, proposal in calibrate, configuration_type is arc, rlevel = 0
#spectrscopic science = instrument in spectroscopy, proposal not in calibrate, configuration_type is science. rlevel = 0

In [ ]:
# first lets filter out spectroscopy frames:
spec_frames = []
for frame in frames:
    if frame["instrument_id"].lower() in Constants.OBSMODE_INSTRUMENTS['spectroscopy']:
        spec_frames.append(frame)

In [ ]:
unique_configurations = set([frame["configuration_type"] for frame in spec_frames])
rprint(f"Unique configurations in spectroscopy frames: {unique_configurations}")

In [ ]:
unique_targets = set([frame["target_name"] for frame in spec_frames])
rprint(f"Unique targets in spectroscopy frames: {unique_targets}")

In [ ]:
unique_proposals = set([frame["proposal_id"] for frame in spec_frames])
rprint(f"Unique proposals in spectroscopy frames: {unique_proposals}")

In [ ]:
def _get_standard_star_spectroscopic_frames(frames):
    return [frame for frame in frames if frame["configuration_type"].lower() == "spectrum" and frame["proposal_id"] == "calibrate" and frame["RLEVEL"] == 0]

def _get_spectroscopic_arc_frames(frames):
    return [frame for frame in frames if frame["configuration_type"].lower() == "arc" and frame["RLEVEL"] == 0]

def _get_spectroscopic_bias_frames(frames):
    return [frame for frame in frames if frame["configuration_type"].lower() == "bias" and frame["RLEVEL"] == 0 and Constants.REQUIRED_CCD_BINNING in frame["target_name"]]

def _get_spectroscopic_flat_frames(frames):
    return [frame for frame in frames if frame["configuration_type"].lower() == "lampflat" and frame["RLEVEL"] == 0 and Constants.REQUIRED_CCD_BINNING in frame["filename"] and "slit" not in frame["filename"]]

def _get_spectroscopic_science_frames(frames):
    return [frame for frame in frames if frame["configuration_type"].lower() == "spectrum" and frame["RLEVEL"] == 0 and frame["proposal_id"] != "calibrate"]

In [ ]:
science_frames = _get_spectroscopic_science_frames(spec_frames)
print(f"Number of spectroscopic science frames: {len(science_frames)}")

In [ ]:
unique_targets = set([frame["target_name"] for frame in science_frames])
rprint(f"Unique targets in spectroscopic bias frames: {unique_targets}")

In [ ]:
test_arcs = [f for f in frames
                if f['configuration_type'] == 'SPECTRUM' and f['proposal_id'] == 'calibrate' and f["reduction_level"] == 0
                and f['instrument_id'].lower() in Constants.OBSMODE_INSTRUMENTS['spectroscopy']]

print(f"Number of spectroscopic arc frames: {len(test_arcs)}")

In [ ]:
for arc in test_arcs:
    print(arc["filename"])

In [ ]:
bias = _get_spectroscopic_bias_frames(spec_frames)
print(f"Number of spectroscopic bias frames: {len(bias)}")

In [ ]:
unique_targets = set([frame["target_name"] for frame in bias])
rprint(f"Unique targets in spectroscopic bias frames: {unique_targets}")

In [ ]:
for bia in bias:
    print(bia["filename"])

In [ ]:
flats = [frame for frame in spec_frames if frame["configuration_type"].lower() == "lampflat" and frame["RLEVEL"] == 0 and Constants.REQUIRED_CCD_BINNING in frame["filename"] and "slit" not in frame["filename"]]

In [ ]:
Constants.REQUIRED_CCD_BINNING

In [ ]:
print(f"Number of spectroscopic arc frames: {len(flats)}")

In [ ]:
for flat in flats:
    print(flat["filename"])

In [ ]:
arcs = [frame for frame in spec_frames if frame["configuration_type"].lower() == "arc" and frame["RLEVEL"] == 0]

In [ ]:
print(f"Number of spectroscopic arc frames: {len(arcs)}")

In [ ]:
for arc in arcs:
    print(arc["filename"])

In [ ]:
#fetch configuration = spectrum and proposal in calibrate

standard_star = [frame for frame in spec_frames if frame["configuration_type"].lower() == "spectrum" and frame["proposal_id"] == "calibrate" and frame["RLEVEL"] == 0]
rprint(f"Number of spectroscopic arc frames: {len(test_arc)}")

In [ ]:
for f in standard_star:
    print(f["filename"])

In [ ]:
target = "ZTF26aaxhpdm"
files = []
for frame in frames:
    if frame['target_name'] == target:
        files.append(frame)
len(files)

In [ ]:
targets = set([frame['target_name'] for frame in files])
insts = set([frame['instrument_id'] for frame in files])
rprint(f"Unique targets: {targets}")
rprint(f"Unique instruments: {insts}")

In [ ]:
times = ["2026-06-20T01:21:20.734", "2026-06-20T01:18:36.328", "2026-06-20T01:10:24.416", "2026-06-19T03:22:32.609", "2026-06-19T03:19:42.407"]
mags = [20.222700000000003, 21.647, 22.0455, 20.42155, 20.700000000000003]
mag_errs = [0.056329075252420605, 0.12330076596160605, 0.06520994486748932, 0.08289556070366495, 0.05236260927417581]
filters = ['z', 'i', 'r', 'z', 'i']
lim_mags = [21.34038232857485, 21.957424442768623, 23.122547769797634, 21.169649999999997, 21.9131]
for time, mag, mag_err, lim_mag, filter in zip(times, mags, mag_errs, lim_mags, filters):
    print(f"{Time(time).mjd},{mag},{mag_err},{lim_mag},ab,sdss{filter}")

In [ ]:
all_targets = set(frame['target_name'] for frame in frames)
print(all_targets)

In [ ]:
download, filenames = [], []
for frame in frames:
    if frame["instrument_id"] in Constants.OBSMODE_INSTRUMENTS['photometry'] or frame["target_name"] == "ZTF26aaxhvtq":
        download.append(frame['url'])
        filenames.append(frame['filename'])

In [ ]:
targets = ["BIAS_344ATTN3_IM_2x2", "ZTF26aaxhpdm", ] 

In [ ]:
for file in filenames:
    print(file)

In [ ]:
import requests, os

output_dir = os.path.expanduser("~/Downloads/lensed_sn_20/raw")
os.makedirs(output_dir, exist_ok=True)

for link, filename in zip(download, filenames):
    r = requests.get(link)
    with open(f"{output_dir}/{filename}", "wb") as f:
        f.write(r.content)
    print(f"Downloaded {filename}")

In [ ]:
for file in files:
    print(file)

In [ ]:
frames = client.get_frames("2025-12-01T00:00:00Z", "2025-12-10T00:00:00Z")
spec_frames = client._get_spectroscopy_science(frames)

In [ ]:
import importlib
import soarpy.lco_client.lco_client as _mod
importlib.reload(_mod)

from soarpy.lco_client.lco_client import LCOClient

client = LCOClient()

In [ ]:
len(frames)

In [ ]:
spec_frames = client._get_spectroscopic_science_frames(frames)
len(spec_frames)

In [ ]:
target = "ZTF26aaxhpdm"

In [ ]:
count = 0
target_names = []
for frame in frames:
    if frame["instrument_id"].lower() in ["ghts_red", "ghts_blue"] and frame["OBSTYPE"] == "SPECTRUM" and frame["reduction_level"] == 0:
        count += 1
        target_names.append(frame["configuration_type"])

print(count)
print(len(set(target_names)))

In [ ]:
congis = set()
for frame in frames:
    congis.add(frame["configuration_type"])

print(congis)

In [ ]:
a = [f for f in frames
                if f['configuration_type'] == 'SPECTRUM' and f["reduction_level"] == 0
                and f['instrument_id'].lower() in Constants.OBSMODE_INSTRUMENTS['spectroscopy']
                and f['proposal_id'] != 'calibrate']
len(a)

In [ ]:
Constants.OBSMODE_INSTRUMENTS['spectroscopy']

In [ ]:
standard_frames = []
for frame in frames:
    if frame["target_name"] == "HR9087":
    # if frame["target_name"] == "HR9087" and frame["OBSTYPE"] == "SPECTRUM":
        standard_frames.append(frame)
        

In [ ]:
instrument_id: ['ghts_red_imager', 'ghts_red', 'triplespec', 'GHTS_RED_IMAGER', 'GHTS_RED']
target_name: ['DFLAT', 'QFLAT', 'BIAS_344ATTN3_IM_2x2', 'BIAS_344ATTN3_SP_2x2', 'ARC', 'DFLAT_OFF', 'DFLAT_ON']
standards = ['HR1996', 'HR9087']


In [ ]:
rlevels = []
for frame in frames:
     rlevels.append(frame["reduction_level"])

print(set(rlevels))

In [ ]:
for frame in frames:
    if frame["reduction_level"] == 63:
        print(frame["filename"])

In [ ]:
BIAS = []
def bias_frames(frames):
    


In [ ]:
INSTRUMENT_ID = [f for f in frames['instrument_id']]

In [ ]:
INSTRUMENT_ID

In [ ]:
skip = {'version_set', 'related_frames'}
unique_vals = {col: df[col].unique().tolist() for col in cols if col not in skip}

In [ ]:
for col in df.columns:
    print(f"Column: {col}")

In [ ]:
INSTRUMENT_ID = unique_vals['instrument_id']
REDUCTION_LEVEL = unique_vals['reduction_level']
SITE_ID = unique_vals['site_id']
TELESCOPE_ID = unique_vals['telescope_id']
PRIMARY_OPTICAL_ELEMENT = unique_vals['primary_optical_element']
CONFIGURATION_TYPE = unique_vals['configuration_type']
FILTER = unique_vals['FILTER']

In [ ]:
for k in unique_vals:
    print(f"{k}: {unique_vals[k]}")

In [ ]:
unique_vals['instrument_id']   # ['ghts_red_imager', 'ghts_blue_imager', ...]
unique_vals['configuration_type']  # ['LAMPFLAT', 'SPECTRUM', 'ARC', ...]

In [ ]:
for col in ['instrument_id', 'configuration_type', 'primary_optical_element', 'reduction_level']:
    print("*" * 40)
    print(df[col].value_counts())

In [ ]:
# peek at the keys available in a frame
frames[0].keys()

In [ ]:
# and one full example to see what the values look like
import json
print(json.dumps(frames[0], indent=2))

In [ ]:
df = pd.DataFrame(frames)

# columns we actually care about — drop the redundant FITS-header duplicates
cols = ['id', 'basename', 'observation_date', 'observation_day', 'proposal_id',
        'instrument_id', 'target_name', 'configuration_type', 'exposure_time',
        'primary_optical_element', 'reduction_level', 'url', 'filename']

df = df[cols]